# 04 — LeRobot Local Inference

Run HuggingFace VLA models directly on GPU — no server. Supports ACT, Pi0,
SmolVLA, Diffusion, XVLA, and any model LeRobot can load.

**Requires**: `pip install "strands-robots[lerobot]"` + GPU

## Architecture

```
Robot observation dict
 → ProcessorBridge.preprocess()     # normalize, device, crop, tokenize
 → LeRobot policy.select_action()   # or predict_action_chunk() for RTC
 → ProcessorBridge.postprocess()     # unnormalize, delta-action
 → list[dict] per-joint actions
```

## Policy Class Resolution

`resolution.py` finds the correct LeRobot class from HF config.json:

```
1. PreTrainedConfig draccus resolution (LeRobot 0.5+)
   ↓ fails?
2. Manual config.json "type" field reading
   ↓ then resolves class via:
3. lerobot.policies.{type}.modeling_{type}  ← LeRobot 0.5+ convention
4. lerobot.policies.{type}                   ← package-level
5. lerobot.policies.factory.get_policy_class ← legacy <0.4
6. PreTrainedPolicy                           ← if concrete
```

A lightweight stub for `lerobot.policies` is injected to avoid importing
the heavy `__init__` (which pulls groot → flash_attn).

In [ ]:
# Resolution by explicit type name (no model download)
try:
    from strands_robots.policies.lerobot_local.resolution import resolve_policy_class_by_name
    ACT = resolve_policy_class_by_name("act")
    print(f"ACT class: {ACT}")
except ImportError as e:
    print(f"lerobot not installed: {e}")

## Loading & Running a Model

```python
from strands_robots import create_policy

# Smart string: auto-detects lerobot_local from HF org
policy = create_policy("lerobot/act_aloha_sim_transfer_cube_human")

# Explicit provider + kwargs
policy = create_policy(
    "lerobot_local",
    pretrained_name_or_path="lerobot/act_aloha_sim_transfer_cube_human",
    device="cuda",
    actions_per_step=1,     # return 1 action per get_actions() call
    use_processor=True,      # load preprocessor/postprocessor pipelines
)

# set_robot_state_keys maps tensor indices → joint names
# If omitted, auto-detects from output_features.action.shape → joint_0, joint_1, ...
policy.set_robot_state_keys([
    "waist", "shoulder", "elbow", "forearm_roll",
    "wrist_angle", "wrist_rotate", "gripper",
])

# Inference
obs = {"observation.state": state_tensor, "observation.image.top": image_array}
actions = policy.get_actions_sync(obs, "pick up the cube")
# actions[0] = {"waist": 0.123, "shoulder": -0.456, ...}
```

## ProcessorBridge

Integrates LeRobot's `DataProcessorPipeline` (preprocessor.json / postprocessor.json).

In [ ]:
from strands_robots.policies.lerobot_local.processor import (
    ProcessorBridge, PREPROCESSOR_CONFIG, POSTPROCESSOR_CONFIG,
)

# Empty bridge — passthrough
bridge = ProcessorBridge()
print(f"active: {bridge.is_active}")           # False
print(f"has_pre: {bridge.has_preprocessor}")   # False
print(f"has_post: {bridge.has_postprocessor}") # False
print(f"info: {bridge.get_info()}")

print(f"\nConfig filenames: {PREPROCESSOR_CONFIG}, {POSTPROCESSOR_CONFIG}")

```python
# Load from a pretrained model (downloads configs from HF)
bridge = ProcessorBridge.from_pretrained(
    "lerobot/act_aloha_sim",
    device="cuda",
    overrides={"image_size": [224, 224]},  # optional step overrides
)

processed_obs = bridge.preprocess(raw_obs, instruction="pick cube")
final_action = bridge.postprocess(raw_action_tensor)
bridge.reset()  # clear stateful step state
```

The bridge builds a full LeRobot `Transition` with complementary data so
VLA tokenizer steps can access the task instruction via the `"task"` key.

## Real-Time Chunking (RTC)

For flow-matching policies (Pi0, SmolVLA), RTC blends action chunks across
inference calls to compensate for compute latency.

```python
policy = create_policy(
    "lerobot_local",
    pretrained_name_or_path="lerobot/pi0_aloha",
    rtc_enabled=True,             # auto-detected from config if None
    rtc_execution_horizon=10,     # timesteps from prefix for guidance
    rtc_max_guidance_weight=10.0, # max correction weight
)
```

Internals:
- `_predict_with_rtc()`: calls `predict_action_chunk()` with `prev_chunk_left_over`
- `_estimate_inference_delay()`: p95 latency → steps consumed during compute
- `_rtc_prev_chunk`: leftover from previous chunk for temporal blending
- Auto-detected from `model.config.rtc_config.enabled`
- Only supported by policies with `rtc_config` (Pi0, SmolVLA), NOT ACT/Diffusion

## VLA Language Tokenization

For VLA models needing language tokens, the policy auto-resolves a tokenizer:

```
1. config.tokenizer_name (explicit)
2. config.vlm_model_name (VLM's tokenizer, e.g. PaliGemma)
3. policy.processor.tokenizer (built-in)
```

Tokens are injected as `observation.language.tokens` + `attention_mask`
into the batch. `_needs_language_tokens()` checks if the model config
declares any language-related input features.

XVLA compat: patches `Florence2LanguageConfig.forced_bos_token_id = None`
for transformers 5.x compatibility.

## Episode Reset

```python
for episode in range(n_episodes):
    policy.reset()  # MUST call — clears:
                    #   - LeRobot internal action queue + temporal ensemble
                    #   - RTC prev_chunk, action_queue, latency_history
                    #   - ProcessorBridge stateful step state
    obs = env.reset()
    for step in range(max_steps):
        actions = policy.get_actions_sync(obs, instruction)
        obs = env.step(actions[0])
```